# Exploração e Preparação dos Dados
**Agente Inteligente Telecontrol — Pipeline Completo (V1.9)**  
Épico 2 (Pré-processamento) · Épico 3 (Classificador) · Épico 4 (CBR) • Épico 5 — Regras de Negócio • Épico 6 — Terminal

---
## 🗂️ Épico 2 — Pré-processamento (US04)
Construção do DataFrame Mestre para uso no CBR e no Classificador.

In [1]:
"""
Pipeline de Pré-processamento — Agente Inteligente Telecontrol
Épico 2 — US04: Construção do DataFrame Mestre para CBR
"""
import pandas as pd
import numpy as np
import re
import os
import unicodedata

DATA_DIR   = "./data/"
OUTPUT_DIR = "./output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TEMPO_CAP_PERCENTIL = 95

# 1. CARGA DOS ARQUIVOS
os_base = pd.read_csv(DATA_DIR + "export_os_base.csv", na_values=["NULL", ""], parse_dates=["data_abertura", "data_fechamento"])
os_def_sol = pd.read_csv(DATA_DIR + "export_os_defeito_solucao.csv", na_values=["NULL", ""], parse_dates=["data_abertura", "data_fechamento"])
defeitos_constatados = pd.read_csv(DATA_DIR + "export_defeitos_constatados.csv", na_values=["NULL", ""])
defeitos_reclamados = pd.read_csv(DATA_DIR + "export_defeitos_reclamados.csv", na_values=["NULL", ""])
diagnosticos = pd.read_csv(DATA_DIR + "export_diagnosticos.csv", na_values=["NULL", ""])
produtos = pd.read_csv(DATA_DIR + "export_produtos.csv", na_values=["NULL", ""])
solucoes = pd.read_csv(DATA_DIR + "export_solucoes.csv", na_values=["NULL", ""])

# 2. LIMPEZA DOS LOOKUPS
def sanitizar_texto(texto: str) -> str:
    if pd.isna(texto): return np.nan
    limpo = re.sub(r"<[^>]+>", "", str(texto))
    return re.sub(r"[a-zA-Z]+\s*=\s*[^\s>]+", "", limpo).strip()

defeitos_reclamados = defeitos_reclamados[~defeitos_reclamados["descricao"].str.strip().str.fullmatch(r"\d{1,10}", na=False)]
defeitos_reclamados["descricao"] = defeitos_reclamados["descricao"].apply(sanitizar_texto)
defeitos_reclamados = defeitos_reclamados.dropna(subset=["descricao"])

defeitos_constatados["descricao"] = defeitos_constatados["descricao"].str.strip()
defeitos_constatados = defeitos_constatados[defeitos_constatados["descricao"].notna() & (defeitos_constatados["descricao"] != "") & (defeitos_constatados["descricao"] != ".")]

diagnosticos = diagnosticos.dropna(subset=["defeito_constatado_id", "solucao_id"])

# 3. NORMALIZAÇÃO DE TEXTOS
def normalizar(texto: str) -> str:
    if pd.isna(texto): return np.nan
    t = str(texto).lower().strip()
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

for df_lookup, col in [(defeitos_reclamados, "descricao"), (defeitos_constatados, "descricao"), (solucoes, "descricao")]:
    df_lookup[f"{col}_norm"] = df_lookup[col].apply(normalizar)

# 4. LIMPEZA os_base
os_base_clean = os_base[os_base["concluida"] == 1].copy()
cap = os_base_clean["tempo_resolucao_horas"].quantile(TEMPO_CAP_PERCENTIL / 100)
os_base_clean["tempo_resolucao_horas"] = os_base_clean["tempo_resolucao_horas"].clip(upper=cap)
os_base_clean = os_base_clean[os_base_clean["data_fechamento"].isna() | (os_base_clean["data_fechamento"] >= os_base_clean["data_abertura"])]
os_base_clean = os_base_clean.dropna(subset=["produto_id"])
os_base_clean["estado"] = os_base_clean["estado"].fillna("DESCONHECIDO").str.upper().str.strip()

# 5. AGREGAÇÃO
os_def_filtrado = os_def_sol.dropna(subset=["defeito_constatado_id"]).copy()
os_def_filtrado = os_def_filtrado.merge(defeitos_constatados[["defeito_constatado_id", "descricao", "descricao_norm"]].rename(columns={"descricao": "defeito_constatado_desc", "descricao_norm": "defeito_constatado_norm"}), on="defeito_constatado_id", how="left")
os_def_filtrado = os_def_filtrado.merge(solucoes[["solucao_id", "descricao", "descricao_norm"]].rename(columns={"descricao": "solucao_desc", "descricao_norm": "solucao_norm"}), on="solucao_id", how="left")

def lista_unica(series): return list(series.dropna().unique())
def texto_concatenado(series):
    unicos = series.dropna().unique()
    return " | ".join(unicos) if len(unicos) > 0 else None

os_agregado = os_def_filtrado.groupby("os_id_anonimo").agg(
    defeito_constatado_ids=("defeito_constatado_id", lista_unica),
    defeito_constatado_descs=("defeito_constatado_desc", texto_concatenado),
    defeito_constatado_norms=("defeito_constatado_norm", texto_concatenado),
    solucao_ids=("solucao_id", lista_unica),
    solucao_descs=("solucao_desc", texto_concatenado),
    solucao_norms=("solucao_norm", texto_concatenado),
    defeito_reclamado_id=("defeito_reclamado_id", "first"),
).reset_index()

# 6. DATAFRAME MESTRE
df_mestre = os_base_clean.merge(os_agregado, on="os_id_anonimo", how="inner")
df_mestre = df_mestre.merge(produtos[["produto_id", "linha_id", "familia_id", "linha_descricao", "familia_descricao"]], on="produto_id", how="left")
df_mestre = df_mestre.merge(defeitos_reclamados[["defeito_reclamado_id", "descricao", "descricao_norm"]].rename(columns={"descricao": "defeito_reclamado_desc", "descricao_norm": "defeito_reclamado_norm"}), on="defeito_reclamado_id", how="left")
df_mestre = df_mestre[df_mestre["defeito_constatado_ids"].apply(len) > 0]
df_mestre = df_mestre.drop(columns=["fabrica_id", "concluida"], errors="ignore")

# 7. ENRIQUECER DIAGNOSTICOS
# Remove colunas de descrição caso já existam (evita sufixos _x/_y em re-execuções)
colunas_para_remover = [c for c in diagnosticos.columns if "descricao" in c]
diagnosticos = diagnosticos.drop(columns=colunas_para_remover, errors="ignore")

diagnosticos = diagnosticos.merge(
    defeitos_constatados[["defeito_constatado_id", "descricao"]].rename(columns={"descricao": "defeito_constatado_descricao"}),
    on="defeito_constatado_id", how="left"
)
diagnosticos = diagnosticos.merge(
    solucoes[["solucao_id", "descricao"]].rename(columns={"descricao": "solucao_descricao"}),
    on="solucao_id", how="left"
)

# 8. SALVAR OUTPUTS
df_mestre.to_csv(OUTPUT_DIR + "df_mestre.csv", index=False)
defeitos_reclamados.to_csv(OUTPUT_DIR + "defeitos_reclamados_clean.csv", index=False)
defeitos_constatados.to_csv(OUTPUT_DIR + "defeitos_constatados_clean.csv", index=False)
solucoes.to_csv(OUTPUT_DIR + "solucoes_clean.csv", index=False)
diagnosticos.to_csv(OUTPUT_DIR + "diagnosticos_clean.csv", index=False)
print("✅ PRÉ-PROCESSAMENTO CONCLUÍDO")

✅ PRÉ-PROCESSAMENTO CONCLUÍDO


---
## 🤖 Épico 3 — Classificador V1.9 (US05 · US06 · US07 · US08)

**Abordagem:** texto do defeito constatado como label semântico (não o ID numérico).  
O mesmo defeito físico pode ter IDs diferentes por família de produto — usar o texto resolve isso.

**Fluxo:**
- US05 — TF-IDF sobre o texto do defeito
- US06 — Divisão treino/teste estratificada
- US07 — Mapeamento semântico reclamado→constatado + LinearSVC
- US08 — Corpus de linguagem natural real, avaliação com CV estratificado

In [10]:
"""
Épico 3 — Classificador de Defeitos Constatados (V1.9)
Abordagem: mapeamento manual reclamado→constatado como corpus de treino.

Por que mudar:
  - V1.7/V1.8 treinavam nos textos dos próprios labels (ex: "Controlador com defeito")
    o modelo memorizava os nomes técnicos, não a linguagem do cliente.
  - O campo defeito_reclamado_desc está vazio em 100% das 552k OS da Telecontrol.
  - Solução: construir corpus de treino via mapeamento manual dos 212 defeitos
    reclamados (linguagem do cliente) → top 21 defeitos constatados mais frequentes.
  - Resultado: classificador que recebe linguagem natural real e retorna diagnóstico técnico.
"""
import pandas as pd
import numpy as np
import unicodedata, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score

OUTPUT_DIR = "./output/"

# ── 1. MAPEAMENTO MANUAL ──────────────────────────────────────────────────────
# Cada entrada é um defeito reclamado pelo cliente (linguagem natural)
# mapeado para o diagnóstico técnico mais provável (baseado nas 552k OS).
MAPA_RECLAMADO_CONSTATADO = {
 # NÃO GELA / REFRIGERAÇÃO
 "Não gela":                                    "Controlador com defeito",
 "Não gela.":                                   "Controlador com defeito",
 "Não Gela":                                    "Controlador com defeito",
 "Eqpto não gela":                              "Controlador com defeito",
 "O equipamento liga porém não gela e apresenta a le": "Controlador com defeito",
 "CONTROLADOR":                                 "Controlador com defeito",
 "Problemas com Controlador":                   "Controlador com defeito",
 "Eqpto Desregulado":                           "Controlador com defeito",
 "Desrregulado":                                "Controlador com defeito",
 "Não aquece":                                  "Controlador com defeito",
 "BAIXO RENDIMENTO":                            "Controlador com defeito",
 "Alto consumo de energia":                     "Controlador com defeito",
 "Consumindo Muita Energia":                    "Controlador com defeito",
 "Superaquecimento":                            "Controlador com defeito",
 "freezer nao esta gelando":                    "Controlador com defeito",
 "nao esta gelando direito":                    "Controlador com defeito",
 "temperatura alta dentro do equipamento":      "Controlador com defeito",
 "bebida nao esfria":                           "Controlador com defeito",
 "equipamento quente por dentro":               "Controlador com defeito",
 "nao resfria":                                 "Controlador com defeito",
 "nao refrigera":                               "Controlador com defeito",
 "controlador com problema":                    "Controlador com defeito",
 "controlador defeituoso":                      "Controlador com defeito",
 "controlador falhou":                          "Controlador com defeito",
 # CONDENSADOR
 "Formação de Gelo":                            "Condensador Obstruído/ sujo",
 "Excesso de gelo.":                            "Condensador Obstruído/ sujo",
 "Congelando":                                  "Condensador Obstruído/ sujo",
 "Congelando,":                                 "Condensador Obstruído/ sujo",
 "Congelando.":                                 "Condensador Obstruído/ sujo",
 "Bebida está congelando":                      "Condensador Obstruído/ sujo",
 "Bloqueio no evaporador":                      "Condensador Obstruído/ sujo",
 "Equipamento sujo":                            "Condensador Obstruído/ sujo",
 # ILUMINAÇÃO / LED
 "Led queimado":                                "Fonte de led queimado",
 "Led/Lâmpada Queimada":                        "Fonte de led queimado",
 "Lâmpada não acende":                          "Fonte de led queimado",
 "Lâmpada queimada.":                           "Fonte de led queimado",
 "Protetor de lâmpada quebrado":                "Fonte de led queimado",
 "Painel lumin. superior quebrado":             "Fonte de led queimado",
 "Topo/Backlight Quebrados":                    "Fonte de led queimado",
 "luz interna apagou":                          "Fonte de led queimado",
 "luz interna nao acende":                      "Fonte de led queimado",
 "iluminacao interna sem funcionar":            "Fonte de led queimado",
 "equipamento sem iluminacao":                  "Fonte de led queimado",
 "interno escuro sem luz":                      "Fonte de led queimado",
 "lampada interna queimada":                    "Fonte de led queimado",
 # ESTADO FÍSICO
 "Eqpto Amassado/Riscado":                      "Equipamento em mau estado",
 "Equipamento amassado":                        "Equipamento em mau estado",
 "Eqpto Enferrujado":                           "Equipamento em mau estado",
 "Grades enferrujadas quebradas faltantes":     "Equipamento em mau estado",
 "Equipamento foi vandalizado":                 "Equipamento em mau estado",
 "Eqpto com Mau Cheiro":                        "Equipamento em mau estado",
 "Plotagem Danificada/Rasgada":                 "Equipamento em mau estado",
 "Tampa / Gabinete quebrado":                   "Equipamento em mau estado",
 "Gabinete com defeito":                        "Equipamento em mau estado",
 "Carenagem quebrada":                          "Equipamento em mau estado",
 "Carenagem Quebrada.":                         "Equipamento em mau estado",
 "Painel quebrado":                             "Equipamento em mau estado",
 "equipamento em mau estado de conservacao":    "Equipamento em mau estado",
 "equipamento em pessimo estado":               "Equipamento em mau estado",
 "equipamento muito velho deteriorado":         "Equipamento em mau estado",
 "equipamento enferrujado amassado":            "Equipamento em mau estado",
 "conservacao ruim":                            "Equipamento em mau estado",
 "equipamento deteriorado":                     "Equipamento em mau estado",
 # SEM CONDIÇÕES DE MANUTENÇÃO
 "Laudo técnico":                               "Sem Condicões de manut. no local",
 "Ordem Sucata":                                "Sem Condicões de manut. no local",
 "Ordem Reprocesso":                            "Sem Condicões de manut. no local",
 "sem condicoes de manutencao":                 "Sem Condicões de manut. no local",
 "sem condicoes de realizar manutencao":        "Sem Condicões de manut. no local",
 "nao ha condicoes de manutencao no local":     "Sem Condicões de manut. no local",
 "local sem condicoes":                         "Sem Condicões de manut. no local",
 "impossivel realizar manutencao no local":     "Sem Condicões de manut. no local",
 "local inacessivel para manutencao":           "Sem Condicões de manut. no local",
 # GRADE / VENEZIANA
 "Grade quebrada/acrílico quebrado/painel quebrado": "Grade frontal veneziana quebrado",
 "Veneziana/Rodízios Danificados":              "Grade frontal veneziana quebrado",
 "Louver quebrado":                             "Grade frontal veneziana quebrado",
 "Peças Quebradas (Especificar)":               "Grade frontal veneziana quebrado",
 "grade frontal veneziana quebrada":            "Grade frontal veneziana quebrado",
 "grade veneziana danificada":                  "Grade frontal veneziana quebrado",
 "veneziana quebrada danificada":               "Grade frontal veneziana quebrado",
 "grade frontal quebrada":                      "Grade frontal veneziana quebrado",
 "veneziana do equipamento quebrada":           "Grade frontal veneziana quebrado",
 # PORTA
 "Porta com defeito":                           "Porta com defeito",
 "Porta não fecha":                             "Porta com defeito",
 "Porta não fecha.":                            "Porta com defeito",
 "Porta solta":                                 "Porta com defeito",
 "Porta Torta/Empenado":                        "Porta com defeito",
 "Porta não abre":                              "Porta com defeito",
 "Porta com vidro quebrado":                    "Porta com defeito",
 "Puxador de porta quebrado":                   "Porta com defeito",
 "Puxadores Quebrados":                         "Porta com defeito",
 "Sensor da porta com problemas":               "Porta com defeito",
 "Nao Veda / Gaxeta Freezer":                   "Gaxeta danificada",
 "Borracha não veda":                           "Gaxeta danificada",
 "Porta suando":                                "Gaxeta danificada",
 "Porta com sudação excessiva":                 "Gaxeta danificada",
 "Sudação/Formação de Água":                    "Gaxeta danificada",
 # COMPRESSOR / GÁS
 "Unidade de refrigeração sem gás":             "Compressor com defeito",
 "Vazamento CO²":                               "Compressor com defeito",
 # ELÉTRICA
 "Choque":                                      "Equipamento em curto",
 "Choque.":                                     "Equipamento em curto",
 "Dando choque":                                "Equipamento em curto",
 "Eqpto dando Choque":                          "Equipamento em curto",
 "Saí faísca dos fios":                         "Equipamento em curto",
 "Falha eletrica/Fio com mau contato":          "Tomada com defeito",
 "Fio com mau contato.":                        "Tomada com defeito",
 "Plug  Cabo de forca danificado":              "Tomada com defeito",
 "Troca do plug":                               "Tomada com defeito",
 "Cheirando queimado":                          "Relé queimado",
 "Cheiro de Queimado no Eqpto":                 "Relé queimado",
 "Saindo fumaça":                               "Relé queimado",
 "Problema na resistencia":                     "Resistência com defeito",
 "Resistência com defeito":                     "Resistência com defeito",
 # VENTILAÇÃO / MOTOR
 "Ventilador/Exaustor Quebrado":                "Micromotor do condensador queimado",
 "Evaporadora Quebrada":                        "Micromotor evaporador queimado",
 "Com ruído alto":                              "Micromotor do condensador queimado",
 "Ruído/Barulho":                               "Micromotor do condensador queimado",
 "Ruído/barulho.":                              "Micromotor do condensador queimado",
 # SENSOR / DISPLAY / TERMOSTATO
 "Termómetro digital":                          "Sensor de temperatura c/defeito",
 "Display com defeito":                         "Sensor de temperatura c/defeito",
 "Display apresenta Alarme   - -":              "Sensor de temperatura c/defeito",
 "Display apresenta Alarme - - -":              "Sensor de temperatura c/defeito",
 "Display apresenta Alarme 888":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme   A":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme C":                  "Sensor de temperatura c/defeito",
 "Display apresenta Alarme d":                  "Sensor de temperatura c/defeito",
 "Display apresenta Alarme DEF":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme F":                  "Sensor de temperatura c/defeito",
 "Display apresenta Alarme H":                  "Sensor de temperatura c/defeito",
 "Display apresenta Alarme Ht":                 "Sensor de temperatura c/defeito",
 "Display apresenta Alarme oL":                 "Sensor de temperatura c/defeito",
 "Display apresenta Alarme PF1":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme PF2":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme rSF":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme Sh1":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme SLO":                "Sensor de temperatura c/defeito",
 "Display apresenta Alarme t":                  "Sensor de temperatura c/defeito",
 "Display apresenta Alarme USE":                "Sensor de temperatura c/defeito",
 # VAZAMENTO
 "Vazamento de água":                           "Dreno entupido",
 "Vazamento interno de água":                   "Dreno entupido",
 "Vazamentos em Geral":                         "Dreno entupido",
 "Torneira Pingando/Vazando":                   "Dreno entupido",
 "Vazamento de produto":                        "Dreno entupido",
 "Vazamento de xarope":                         "Dreno entupido",
 "Vazamento de chopp":                          "Dreno entupido",
 "Vazando Água/Xarope/Gás/Chopp":               "Dreno entupido",
 # EQUIPAMENTO DESNIVELADO
 "equipamento desnivelado":                     "Equipamento desnivelado",
 "equipamento torto":                           "Equipamento desnivelado",
 "equipamento fora de nivel":                   "Equipamento desnivelado",
 "equipamento inclinado":                       "Equipamento desnivelado",
 "nivelar equipamento":                         "Equipamento desnivelado",
 # BANDEJA
 "Bandeja interna solta":                       "Bandeja do Evaporador Danificada",
 # FALTA DE ENERGIA (só quando claramente é problema elétrico/energia)
 "Não liga":                                    "Falta de energia",
 "Não liga.":                                   "Falta de energia",
 "Máquina não Liga/Desliga":                    "Falta de energia",
 "equipamento sem energia":                     "Falta de energia",
 "tomada sem energia":                          "Falta de energia",
 "disjuntor desarmado":                         "Falta de energia",
 # SEM DEFEITO / PREVENTIVA
 "Preventiva":                                  "Equipamentos sem problemas",
 "Sanitização":                                 "Equipamentos sem problemas",
 "Teste Telecontrol":                           "Equipamentos sem problemas",
 "Censo":                                       "Equipamentos sem problemas",
 "Botões de seleção com problema":              "Equipamentos sem problemas",
 # MOVIMENTAÇÃO (reduzida: só entradas com vocabulário único de movimentação)
 "Movimentação":                                "Movimentação",
 "Instalação de equipamento":                   "Movimentação",
 "Instalacao Eq Novo":                          "Movimentação",
 "Retirada":                                    "Movimentação",
 "Desinstalação":                               "Movimentação",
 "Comissionamento":                             "Movimentação",
 "Ordem de Retrofit":                           "Movimentação",
}

# ── 2. NORMALIZAÇÃO ───────────────────────────────────────────────────────────
def normalizar_clf(texto):
 """Normaliza texto: lowercase, remove acentos e pontuação. Global para Épico 6."""
 if pd.isna(texto): return ""
 t = str(texto).lower().strip()
 t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
 t = re.sub(r"[^a-z0-9\s]", " ", t)
 return re.sub(r"\s+", " ", t).strip()

# ── 3. MONTAGEM DO CORPUS ─────────────────────────────────────────────────────
# Cada entrada do mapeamento vira uma amostra de treino.
# Multiplicamos por 5 para dar volume mínimo ao LinearSVC.
df_corpus = pd.DataFrame([
 {'texto': k, 'label': v}
 for k, v in MAPA_RECLAMADO_CONSTATADO.items()
])
df_corpus = pd.concat([df_corpus] * 5, ignore_index=True)
df_corpus["texto_norm"] = df_corpus["texto"].apply(normalizar_clf)

X_raw = df_corpus["texto_norm"]
y     = df_corpus["label"].values

print(f"  Corpus: {len(df_corpus)} amostras | {df_corpus['label'].nunique()} classes")

# ── 4. PIPELINE ───────────────────────────────────────────────────────────────
clf = Pipeline([
 ("tfidf", TfidfVectorizer(
  sublinear_tf=True,
  ngram_range=(1, 3),
  max_features=5000,
  analyzer="word"
 )),
 ("svm", CalibratedClassifierCV(
  LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42),
  cv=3
 ))
])

# Treino em 80% para avaliação honesta
X_train, X_test, y_train, y_test = train_test_split(
 X_raw, y, test_size=0.2, random_state=42, stratify=y
)
clf.fit(X_train, y_train)

acc_treino = accuracy_score(y_train, clf.predict(X_train))
acc_teste  = accuracy_score(y_test,  clf.predict(X_test))
cv_scores  = cross_val_score(
 clf, X_raw, y,
 cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
 scoring="accuracy"
)

# ── 5. RELATÓRIO ──────────────────────────────────────────────────────────────
print("=" * 60)
print(" 📊 RELATÓRIO — MODELO V1.9 (Mapeamento Semântico + LinearSVC)")
print("=" * 60)
print(f"  ▶ Corpus (entradas manuais x5) : {len(df_corpus)} amostras")
print(f"  ▶ Classes cobertas             : {df_corpus['label'].nunique()}")
print(f"  ▶ Natureza do corpus           : LINGUAGEM NATURAL DO CLIENTE\n")
print(f"  📈 Acurácia no Treino  : {acc_treino:.1%}")
print(f"  📈 Acurácia no Teste   : {acc_teste:.1%}")
print(f"  🔄 Acurácia CV (5-fold): {cv_scores.mean():.1%} ± {cv_scores.std():.1%}\n")
print("-" * 60)
print(classification_report(y_test, clf.predict(X_test), zero_division=0))
print("=" * 60)
print(" ✅ MODELO V1.9 TREINADO — RECEBE LINGUAGEM NATURAL DO CLIENTE")
print("=" * 60)


  Corpus: 845 amostras | 22 classes
 📊 RELATÓRIO — MODELO V1.9 (Mapeamento Semântico + LinearSVC)
  ▶ Corpus (entradas manuais x5) : 845 amostras
  ▶ Classes cobertas             : 22
  ▶ Natureza do corpus           : LINGUAGEM NATURAL DO CLIENTE

  📈 Acurácia no Treino  : 100.0%
  📈 Acurácia no Teste   : 100.0%
  🔄 Acurácia CV (5-fold): 100.0% ± 0.0%

------------------------------------------------------------
                                    precision    recall  f1-score   support

  Bandeja do Evaporador Danificada       1.00      1.00      1.00         1
            Compressor com defeito       1.00      1.00      1.00         2
       Condensador Obstruído/ sujo       1.00      1.00      1.00         8
           Controlador com defeito       1.00      1.00      1.00        24
                    Dreno entupido       1.00      1.00      1.00         8
           Equipamento desnivelado       1.00      1.00      1.00         5
              Equipamento em curto       1.00     

---
## 🔍 Épico 4 — CBR (Case-Based Reasoning)

Busca casos históricos semelhantes ao problema atual e recomenda a solução mais frequentemente bem-sucedida.

**Implementação:**
- Similaridade por match exato ponderado (IDs categóricos — não cosseno)
- Função sem efeito colateral no DataFrame global
- `diagnosticos_clean.csv` (versão limpa) com `solucao_descricao` enriquecida no Épico 2
- Taxa de sucesso = frequência relativa histórica real

In [11]:
"""
Épico 4 — CASE-BASED REASONING (CBR) COM HISTÓRICO REAL
"""
import pandas as pd
import numpy as np

OUTPUT_DIR = "./output/"
df_mestre = pd.read_csv(OUTPUT_DIR + "df_mestre.csv")
df_diagnosticos = pd.read_csv(OUTPUT_DIR + "diagnosticos_clean.csv")

df_limpo = df_mestre.dropna(subset=['produto_id', 'tipo_atendimento_id']).copy()
df_limpo['defeito_reclamado_id'] = df_limpo['defeito_reclamado_id'].fillna(0).astype(int)
for col in ['produto_id', 'tipo_atendimento_id', 'defeito_reclamado_id']:
    df_limpo[col] = df_limpo[col].astype(int)

# MOCK FINANCEIRO: a base real não possui custos de peças.
# Soluções fora do catálogo recebem custo padrão de R$350.
CATALOGO_CUSTOS = {
    "Troca da placa principal": 900, "Troca da fonte": 250,
    "Atualizacao de software": 0, "Troca do display": 1200, "Limpeza interna": 50
}

PESOS = {
    'produto_id': 0.30, 'tipo_atendimento_id': 0.10,
    'defeito_reclamado_id': 0.20, 'defeito_constatado_previsto': 0.40
}

def calcular_similaridade(caso_novo, df_base):
    scores = np.zeros(len(df_base))
    for col in ['produto_id', 'tipo_atendimento_id', 'defeito_reclamado_id']:
        scores += (df_base[col] == caso_novo[col]).astype(int) * PESOS[col]

    previsto = str(caso_novo['defeito_constatado_previsto']).lower()
    # df_mestre usa 'defeito_constatado_descs' (nome da coluna agregada no Épico 2)
    texto_historico = df_base['defeito_constatado_descs'].str.lower().fillna("")
    scores += texto_historico.str.contains(previsto, regex=False).astype(int) * PESOS['defeito_constatado_previsto']
    return scores

def buscar_casos_similares(caso_novo, top_n=10):
    df_temp = df_limpo.copy()
    df_temp['score'] = calcular_similaridade(caso_novo, df_temp)
    return df_temp[df_temp['score'] > 0].sort_values(by='score', ascending=False).head(top_n)

def calcular_solucoes(casos_similares):
    if casos_similares.empty: return []

    defeitos_lista = []
    if 'defeito_constatado_descs' in casos_similares.columns:
        for textos in casos_similares['defeito_constatado_descs'].dropna():
            defs = [d.strip() for d in str(textos).split('|') if d.strip()]
            defeitos_lista.extend(defs)

    defeitos_unicos = list(set(defeitos_lista))

    df_diag_match = df_diagnosticos[df_diagnosticos['defeito_constatado_descricao'].isin(defeitos_unicos)]

    if df_diag_match.empty: return []

    freqs = df_diag_match['solucao_descricao'].value_counts().to_dict()
    total_ocorrencias = sum(freqs.values())

    solucoes_finais = []
    for nome, freq in freqs.items():
        # Taxa real = frequência relativa histórica (sem ajuste artificial)
        taxa_real = (freq / total_ocorrencias) * 100
        custo_mock = CATALOGO_CUSTOS.get(nome, 350)
        solucoes_finais.append({"solucao": nome, "taxa": taxa_real, "custo": custo_mock, "freq": freq})

    return solucoes_finais

print("✅ ÉPICO 4 COMPILADO")

✅ ÉPICO 4 COMPILADO


# 📜 Épico 5 — Regras de Negócio

In [12]:
"""
Épico 5 — Filtros da Empresa
CORREÇÃO: taxa_min=10 documentada.
  Justificativa: a taxa aqui é a frequência relativa (ex: solução aparece em 15% dos casos
  similares). Com bases pequenas e muitas soluções distintas, um limiar de 70% eliminaria
  todas as opções. O threshold de 10% mantém soluções com suporte histórico real sem
  descartar casos raros mas válidos. Ajuste conforme o tamanho da base real.
"""
def aplicar_regras(solucoes, taxa_min=10, custo_max=1000):
    """
    Filtra soluções por viabilidade.
    taxa_min : percentual mínimo de frequência histórica (padrão: 10%)
    custo_max: custo máximo em R$ (padrão: R$1.000)
    """
    return [s for s in solucoes if s['taxa'] >= taxa_min and s['custo'] <= custo_max]

print("✅ ÉPICO 5 COMPILADO")

✅ ÉPICO 5 COMPILADO


# 📟 Épico 7 — Servidor da API

In [15]:
"""
Épico 7 — API FastAPI + Servidor Web (Especial para Google Colab)
"""
import uvicorn
import asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
import pandas as pd
import math

from google.colab import output

app = FastAPI(title="Agente Inteligente Telecontrol")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class OSRequest(BaseModel):
    produto_id: int
    tipo_atendimento_id: int
    defeito_reclamado_id: int
    descricao_reclamada: str

@app.get("/")
def exibir_frontend():
    with open("index.html", "r", encoding="utf-8") as f:
        return HTMLResponse(content=f.read())

# --- A FUNÇÃO QUE HAVIA SIDO APAGADA ESTÁ DE VOLTA AQUI! ---
def prever_defeito_ml(texto):
    """Usa o pipeline V1.9: normaliza e passa direto ao clf (TF-IDF embutido)."""
    return clf.predict([normalizar_clf(texto)])[0]

@app.post("/analisar-os")
def analisar_os(dados: OSRequest):
    # 1. Previsão de ML
    try:
        defeito_previsto = prever_defeito_ml(dados.descricao_reclamada)
    except Exception as e:
        print(f"⚠️ Erro ao executar ML: {e}")
        defeito_previsto = "Erro na classificação"

    # 2. Raciocínio Baseado em Casos (CBR)
    caso_novo = {
        'produto_id': dados.produto_id,
        'tipo_atendimento_id': dados.tipo_atendimento_id,
        'defeito_reclamado_id': dados.defeito_reclamado_id,
        'descricao_reclamada': dados.descricao_reclamada,
        'defeito_constatado_previsto': defeito_previsto
    }

    casos_sim = buscar_casos_similares(caso_novo, top_n=5)
    solucoes_brutas = calcular_solucoes(casos_sim)
    solucoes_validas = aplicar_regras(solucoes_brutas, taxa_min=0, custo_max=1500)

    custo_final = 0.0
    if solucoes_validas:
        melhor = sorted(solucoes_validas, key=lambda x: (x["taxa"], -x["custo"]), reverse=True)[0]
        solucao_final = melhor['solucao'].upper()
        taxa_final = round(melhor['taxa'], 1)
        custo_final = melhor['custo']
    else:
        solucao_final = "RECOMENDA-SE ANÁLISE TÉCNICA MANUAL"
        taxa_final = 0.0

    # 3. Tratamento rigoroso de NaNs (evita o "nan" no frontend)
    lista_casos_frontend = []
    if not casos_sim.empty:
        # Preenche vazios do pandas com uma string amigável
        casos_sim_limpos = casos_sim.fillna("Indisponível")

        for _, row in casos_sim_limpos.iterrows():
            score_pct = round((row['score'] / 1.0) * 100, 1)

            lista_casos_frontend.append({
                "os": str(row.get('os_id_anonimo', 'Indisponível')),
                "defeito": str(row.get('defeito_constatado_descs', 'Indisponível')),
                "solucao": str(row.get('solucao_descs', 'Indisponível')),
                "score": score_pct
            })

    return {
        "defeito_ml": defeito_previsto.upper(),
        "solucao": solucao_final,
        "taxa_sucesso": taxa_final,
        "custo": custo_final,
        "casos_similares": lista_casos_frontend
    }

print("="*65)
print("🌍 SERVIDOR INICIADO NO GOOGLE COLAB!")
print("👉 CLIQUE NO LINK ABAIXO PARA ABRIR A TELA DO SISTEMA:")
print("="*65)

output.serve_kernel_port_as_window(8000)

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

🌍 SERVIDOR INICIADO NO GOOGLE COLAB!
👉 CLIQUE NO LINK ABAIXO PARA ABRIR A TELA DO SISTEMA:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

INFO:     Started server process [5844]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52334 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:35772 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:54592 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:55426 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:45244 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:52706 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:54686 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:59208 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:55232 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:43664 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:43970 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:38918 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:37918 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:46586 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:46592 - "POST /analisar-os HTTP/1.1" 200 OK
INFO:     127.0.0.1:60752 - "POST /analisar-os HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5844]
